<a href="https://colab.research.google.com/github/DevisriprasadSoma/MLflyrank/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DevisriprasadSoma/MLflyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The action queue prioritizes pages for human review using the Logistic Regression decline-risk score from the validated development workflow. A higher score means the page is ranked higher for review; it does not mean that decline is certain.

Each queued page receives a human-readable reason code based on observable signals such as freshness, search position, engagement, and content depth. The recommended action is intended to guide review rather than automatically change or publish content.

The highest-ranked pages should be reviewed first because they combine stronger model-based decline signal with observable content or performance gaps.


In [6]:
# ============================================
# SECTION 1 — RANKED ACTIONS + REASON CODES
# ============================================

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

repo = "/content/flyrank-ml-internship-starter"
raw_path = os.path.join(repo, "data/raw/content_refresh_anonymized.csv")

if not os.path.exists(raw_path):
    !git clone -q https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

df = pd.read_csv(raw_path)

feature_cols = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "word_count",
    "char_count", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

model_df = df[feature_cols + ["trend_direction"]].dropna().copy()

model_df["target"] = (
    model_df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

X = model_df[feature_cols]
y = model_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

model_df["decline_risk_score"] = model.predict_proba(X)[:, 1]

# Reason codes
def reason_code(row):
    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("STALE_CONTENT")

    if row["avg_position"] >= 15:
        reasons.append("SEARCH_POSITION")

    if row["engagement_rate"] < model_df["engagement_rate"].median():
        reasons.append("LOW_ENGAGEMENT")

    if row["word_count"] < model_df["word_count"].median():
        reasons.append("CONTENT_DEPTH")

    return " + ".join(reasons[:2]) if reasons else "MODEL_RISK"

queue = model_df.copy()
queue["reason_code"] = queue.apply(reason_code, axis=1)

queue["recommended_action"] = np.select(
    [
        queue["reason_code"].str.contains("STALE_CONTENT"),
        queue["reason_code"].str.contains("SEARCH_POSITION"),
        queue["reason_code"].str.contains("LOW_ENGAGEMENT"),
        queue["reason_code"].str.contains("CONTENT_DEPTH")
    ],
    [
        "Review freshness and update outdated information",
        "Review search intent, structure, and on-page optimization",
        "Review content relevance and user engagement",
        "Review content completeness and depth"
    ],
    default="Human review of page performance"
)

queue = queue.sort_values(
    "decline_risk_score", ascending=False
).reset_index(drop=True)

queue["priority_rank"] = np.arange(1, len(queue) + 1)

output_cols = [
    "priority_rank",
    "decline_risk_score",
    "reason_code",
    "recommended_action"
]

action_queue = queue[output_cols].head(50)

print("Ranked action queue created.")
print("Rows in queue:", len(action_queue))
print("\nTop 10 actions:")
display(action_queue.head(10))

Ranked action queue created.
Rows in queue: 50

Top 10 actions:


,priority_rank,decline_risk_score,reason_code,recommended_action
0,1,1.000000,MODEL_RISK,Human review of page performance
1,2,1.000000,MODEL_RISK,Human review of page performance
2,3,0.999985,MODEL_RISK,Human review of page performance
3,4,0.999600,MODEL_RISK,Human review of page performance
4,5,0.999373,SEARCH_POSITION,"Review search intent, structure, and on-page o..."
5,6,0.998450,MODEL_RISK,Human review of page performance
6,7,0.997752,MODEL_RISK,Human review of page performance
7,8,0.997702,MODEL_RISK,Human review of page performance
8,9,0.997507,MODEL_RISK,Human review of page performance
9,10,0.994401,MODEL_RISK,Human review of page performance


## 2. Intended use and limits

The playbook is intended to help content teams prioritize pages for human review. The ranked queue can be used to identify pages that may deserve attention based on the model score and observable content signals.

The output is decision-support, not an automatic content management system. The model was evaluated on development data, and performance decreased under client-grouped validation in ML-09. Therefore, the scores should be treated as directional rather than as guaranteed predictions of future decline.

The playbook should not be used as the sole basis for deleting, rewriting, redirecting, or publishing content. Final decisions should consider search intent, business importance, factual accuracy, current content quality, and human judgment.


In [7]:
print("Intended use: prioritize pages for human review.")
print("Use: directional decision-support for content actions.")
print("Limit: scores are not guaranteed future or production predictions.")
print("Final content decisions require human review.")

Intended use: prioritize pages for human review.
Use: directional decision-support for content actions.
Limit: scores are not guaranteed future or production predictions.
Final content decisions require human review.


## 3. Human review + the no-go list

Before taking action on a ranked page, a human reviewer should check the page's search intent, factual accuracy, content quality, business importance, recent changes, and whether the model reason matches the actual page situation.

The ranked score should be used to prioritize review, not to make the final decision.

### No-go cases

The system should not automatically:

* Delete or redirect a page based only on the model score.
* Rewrite or publish content without human approval.
* Make factual, legal, medical, or sensitive claims automatically.
* Treat a high decline-risk score as proof that a page will decline.
* Override business or editorial decisions using the model score alone.


In [8]:
print("Human review is required before content action.")
print("No-go: automatic deletion, redirects, rewriting, publishing, or factual decisions.")
print("Model scores are prioritization signals, not final decisions.")

Human review is required before content action.
No-go: automatic deletion, redirects, rewriting, publishing, or factual decisions.
Model scores are prioritization signals, not final decisions.


## 4. Monitoring / retrain triggers

The playbook should be reviewed periodically to check whether its recommendations remain useful. Monitoring should focus on changes in model performance, data patterns, and the usefulness of the ranked actions.

Retraining should be considered if measured ranking performance declines consistently, the distribution of important input features changes substantially, or content and search behavior changes enough that the current training data no longer represents the pages being reviewed.

A single unusual result should not automatically trigger retraining. Changes should be reviewed by a human before updating the model.


In [9]:
print("Monitor: ranking performance, feature distributions, and recommendation usefulness.")
print("Retrain trigger: sustained performance decline or meaningful data/distribution change.")
print("Human review is required before retraining.")

Monitor: ranking performance, feature distributions, and recommendation usefulness.
Retrain trigger: sustained performance decline or meaningful data/distribution change.
Human review is required before retraining.


## 5. Exports for the paper

The ranked action queue is exported as a CSV so the paper can reuse the same recommendations generated by this notebook. A metrics JSON is also saved with the validation results used to describe the model.

The queue is an analysis output for research and decision-support. It is not treated as a production data feed.


In [10]:
# ============================================
# SECTION 5 — EXPORTS FOR THE PAPER
# ============================================

import os
import json

output_dir = "/content/MLflyrank/work/outputs"
os.makedirs(output_dir, exist_ok=True)

# Export ranked action queue
queue_path = os.path.join(output_dir, "ml10_ranked_action_queue.csv")
action_queue.to_csv(queue_path, index=False)

# Export validation metrics
metrics = {
    "ml08_stratified_roc_auc": 0.6777,
    "ml08_stratified_average_precision": 0.6945,
    "ml09_client_grouped_roc_auc": 0.5950,
    "ml09_client_grouped_average_precision": 0.5939
}

metrics_path = os.path.join(output_dir, "ml10_validation_metrics.json")

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Exports completed.")
print("Queue:", queue_path)
print("Metrics:", metrics_path)
print("Queue rows:", len(action_queue))

Exports completed.
Queue: /content/MLflyrank/work/outputs/ml10_ranked_action_queue.csv
Metrics: /content/MLflyrank/work/outputs/ml10_validation_metrics.json
Queue rows: 50


## Self-check

Before you submit, confirm each line honestly:

* ☑ Every section above is filled — markdown thinking AND the code that backs it
* ☑ The notebook runs top to bottom with no errors (Runtime → Run all)
* ☑ No client names, URLs, or private queries anywhere
* ☑ My claims use careful words: observed, measured, directional, decision-support
* ☑ The ranked action queue and validation metrics are exported to `work/outputs/`
* ☑ Committed to my repo under `work/notebooks/` — then submit the repo URL on the card.
